# Who Goes There: the final scoring on Kaggle, once

Scores every checkpoint of `docs/ledger.md` (the small CNN, and ResNet-18 arm A and arm B for each
reported run seed) on cis_test and trans_test with `trailcam.evaluate.score_test`, once (items 9.01
and 9.03; the protocol is `docs/plan.md` section 7). This is the user's single final scoring, item
9.02. The code, `config.yaml`, the ledger and the test ledger all come from one clone of the
public repo at the commit `COMMIT`.

The notebook is separate from the training and evaluation notebooks so the test sets can never be
touched by accident: those never name a test split, and this one does nothing but the final
scoring. It never trains and never scores val or trans_val. One code cell runs the scoring
(section 6), after the go-ahead cell (section 5); no cell before it loads a checkpoint's weights or
reads a test image.

## This runs once

A test set is a one-shot measurement: once a test score has been seen, every later choice is tuned
on it a little. So every checkpoint is scored together, once. `score_test.py` refuses a test ledger
that already holds a row, but a fresh clone at the same `COMMIT` holds that commit's empty test
ledger. So section 3 also reads `origin/main`'s `docs/test-ledger.md` and refuses if it holds any
row: once the rows are pasted into `docs/test-ledger.md`, committed and pushed to `main` on
GitHub, the notebook stops in seconds, whatever `COMMIT` and `APPROVED_BY` say. Between the run
and that push, only `APPROVED_BY` stands between the notebook and a second scoring.

- `APPROVED_BY` starts empty. With it empty, the notebook runs every check and then refuses the
  scoring with a message that the user's go-ahead is needed (section 5). That is the rehearsal.
- Never Save & Run All with `APPROVED_BY` filled in until the user means it.
- Right after the one run, set `APPROVED_BY` back to `""` in the Kaggle copy and save it, and
  commit and push the rows to `main` on GitHub.

## How to run on Kaggle

1. Import this notebook into Kaggle: File > Import Notebook, with its GitHub link
   (https://github.com/codx999/who-goes-there/blob/main/notebooks/kaggle_score_test.ipynb).
2. Inputs. Add Input > Notebook: the output of the packing notebook of item 6.08 (`cct20_512.tar`),
   as for training; the default `CACHE_ARCHIVE` is the user's. Add Input > Dataset: the private
   dataset `trailcam-checkpoints` holding the 7 `.pt` files of the ledger's runs (each training
   version kept its checkpoint under `/kaggle/working/models/`). With `CHECKPOINTS_ROOT = None`,
   section 4 finds the one folder named `trailcam-checkpoints` under `/kaggle/input`; if it finds
   none or several, it lists the folders holding `.pt` files, and `CHECKPOINTS_ROOT` is set to the
   right one. Do not attach the loose `cct20-512` dataset: section 3 refuses it in an interactive
   session, as the training notebook does.
3. Keep exactly one parameter cell, the first code cell: delete any copy of it left by an earlier
   import or edit, since a later cell's assignment silently overrides the first. Set `COMMIT` to the
   full sha of the `main` commit the coordinator gives, at or after the merge of item 9.03: its
   `docs/ledger.md` holds the 7 rows and its `docs/test-ledger.md` none. The default is the `main`
   commit this notebook was written against. `RUN_SEEDS` is `(0, 1, 2)`, the run seeds the
   experiment reports (`--run-seeds`); it changes only by the user's decision. Leave `APPROVED_BY`
   empty.
4. Session options: Accelerator GPU T4 x2 (the scoring uses the first T4) and Internet on (the
   clone and pip).
5. Rehearse: with `APPROVED_BY` empty, Run All. Sections 1 to 4 run (parameters, clone and install,
   the code, the scoring not yet recorded on `main`, the GPU, unpacking and the cache check, the
   checkpoints against the ledger), then
   section 5 stops the run with the go-ahead message. No checkpoint is loaded and no test image is
   read. Then stop the session so no GPU time is idle.
6. The one run, when the user means it: set `APPROVED_BY` to the name of who gave the go-ahead,
   then Save Version > Save & Run All (Commit). The version repeats every check in a fresh session,
   then scores. Its Logs tab shows each checkpoint's test ledger row as soon as that checkpoint is
   scored. As for training, open the Logs tab about 10 minutes after submitting: if it is still
   empty and the version still shows "Starting...", cancel it and submit again (no cell has
   started, so nothing has been scored).
7. After it: paste the rows section 7 prints into `docs/test-ledger.md`, once (the clone's copy in
   the version's output holds the same rows), and commit and push them to `main` on GitHub: from
   then on section 3 refuses every run. Download `score.zip` from the version's Output (the
   headline, experiment and slice tables, the per-checkpoint metrics and figures, the feature
   maps), set `APPROVED_BY` back to `""`, then do item 9.02's other steps. If the scoring reports a
   LEAKAGE ALARM, stop and report it before anything else (R16).

## If it stops part way

If the scoring cell (section 6) stops before it prints `appended ... rows`, nothing was appended to
the test ledger: run again from the start with a new `--out`, the recovery `score_test.py`
documents. A new version (Save & Run All) starts with an empty `/kaggle/working`, so `OUT` is a new
folder; in an interactive session, restart it (Run > Factory reset) and set `OUT` to a new folder
first, for example `WORK / "score-2"` (`score_test.py` refuses a non-empty `--out`). The
checkpoints are pinned by sha256, so the scores are the same, and every output covers every
checkpoint. If it stopped after `appended ... rows`, the scoring has run: never run it again. The
rows are in the log and in the clone's `docs/test-ledger.md`, the outputs under `OUT`.

Commit this notebook with its outputs cleared (Edit > Clear All Outputs); it prints counts,
scores, run ids and hashes only, never images or metadata records.

Data: Caltech Camera Traps (CCT20), licence
[CDLA-Permissive 1.0](https://cdla.dev/permissive-1-0/). Please cite: Sara Beery, Grant Van
Horn, Pietro Perona. "Recognition in Terra Incognita." Proceedings of the European Conference
on Computer Vision (ECCV), 2018. https://arxiv.org/abs/1807.04975. Code: MIT.

In [ ]:
# Parameters: the only parameter cell (How to run, step 3). APPROVED_BY stays "" until the go-ahead.
from pathlib import Path

COMMIT = "b64c48691c4df3f3c2be5acf2cd33a3329104b4e"  # the main commit the coordinator gives
CACHE_ARCHIVE = Path("/kaggle/input/notebooks/codx888/notebook2b5a4bf6ad/cct20_512.tar")
CHECKPOINTS_ROOT = None  # None: the one folder named CHECKPOINTS_DATASET under INPUT; or a Path
APPROVED_BY = ""  # empty: every check runs, then the scoring is refused (the rehearsal)
RUN_SEEDS = (0, 1, 2)  # the ResNet-18 run seeds the experiment reports (--run-seeds)

REPO_URL = "https://github.com/codx999/who-goes-there"  # the public repo
INPUT = Path("/kaggle/input")  # where Kaggle mounts the attached inputs
CHECKPOINTS_DATASET = "trailcam-checkpoints"  # the private dataset of the ledger's .pt files
WORK = Path("/kaggle/working")  # kept as the version's output
CLONE = WORK / "who-goes-there"
CONFIG_FILE = CLONE / "config.yaml"
LEDGER = CLONE / "docs" / "ledger.md"
TEST_LEDGER = CLONE / "docs" / "test-ledger.md"  # score_test appends the rows to this copy
ASSETS = CLONE / "assets"  # the photos of the Grad-CAM overlays
OUT = WORK / "score"  # --out: a new folder for every run (If it stops part way)
LOCAL_CACHE = Path("/tmp/trailcam-cache")  # the unpacked archive: local disk, not output
MAIN_TEST_LEDGER = Path("/tmp/trailcam-main-test-ledger.md")  # origin/main's, read in section 3

## 1. Check the parameters and pin one GPU

Fails in seconds, before anything is cloned or installed, and prints the parameters in effect (a
second parameter cell would show here) and whether this run is the rehearsal (`APPROVED_BY`
empty) or the final scoring. `CUDA_VISIBLE_DEVICES=0` is set here, before torch is first imported
(by the `trailcam` import of section 3), so torch, and the scoring's own Python process, which
inherits it, see the first T4 of GPU T4 x2 only.

In [ ]:
import os
import re
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4 of the two; torch is not imported yet
print(f"Python {sys.version.split()[0]}")
if not re.fullmatch(r"[0-9a-f]{40}", COMMIT):
    raise ValueError(f"COMMIT must be a full 40-character commit sha, got {COMMIT!r}")
if not isinstance(APPROVED_BY, str):
    raise ValueError(f'APPROVED_BY must be a string ("" until the go-ahead), got {APPROVED_BY!r}')
if not RUN_SEEDS or not all(isinstance(seed, int) for seed in RUN_SEEDS):
    raise ValueError(f"RUN_SEEDS must hold the reported seeds, e.g. (0, 1, 2), got {RUN_SEEDS!r}")
print(f"COMMIT {COMMIT}, RUN_SEEDS {RUN_SEEDS}, APPROVED_BY {APPROVED_BY!r}")
print(f"CACHE_ARCHIVE {CACHE_ARCHIVE}, CHECKPOINTS_ROOT {CHECKPOINTS_ROOT}")
if APPROVED_BY.strip():
    print(f"APPROVED_BY is filled in: this run is the one final scoring ({APPROVED_BY.strip()})")
else:
    print("APPROVED_BY is empty: the rehearsal; every check runs, then section 5 stops the run")

## 2. Clone the repo at `COMMIT` and install `trailcam` from the clone

`config.yaml`, `docs/ledger.md`, `docs/test-ledger.md` and `assets/` are not in the wheel, so
`score_test.py` is given all four from this clone (`--config`, `--ledger`, `--test-ledger`,
`--assets`): code, configs, ledger and test ledger come from one commit. This is the project's
one `pip install` (the notebook exception in `CLAUDE.md`). The package's dependency floors lie
below Kaggle's versions, so pip keeps the preinstalled torch, torchvision, Pillow, timm and
PyYAML; the cell checks that it did.

The clone may differ from `COMMIT` in `docs/test-ledger.md` alone, the file the scoring appends
to: a session that has already scored then reaches `score_test.py`'s run-once refusal in section 4
and is never sent to a fresh clone, whose test ledger would be empty again.

In [ ]:
import subprocess

if CLONE.exists():  # a second run in one interactive session reuses the clone
    print(f"reusing the clone at {CLONE}")
else:
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(CLONE)], check=True)
    subprocess.run(["git", "-C", str(CLONE), "checkout", "--quiet", "--detach", COMMIT], check=True)
head = subprocess.run(
    ["git", "-C", str(CLONE), "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
changed = subprocess.run(
    ["git", "-C", str(CLONE), "status", "--porcelain", "--untracked-files=no"],
    check=True,
    capture_output=True,
    text=True,
).stdout.split("\n")
changed = [line for line in changed if line and not line.endswith(" docs/test-ledger.md")]
if head != COMMIT or changed:
    raise RuntimeError(
        f"{CLONE} is at {head} with {len(changed)} changed files, not a clean {COMMIT}: "
        "restart the session (Run > Factory reset) and run again"
    )
print(f"commit {head}")

In [ ]:
from importlib.metadata import version

KEPT = ("torch", "torchvision", "Pillow", "timm", "PyYAML")  # Kaggle's preinstalled builds
preinstalled = {name: version(name) for name in KEPT}
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(CLONE)], check=True)
after_install = {name: version(name) for name in KEPT}
if after_install != preinstalled:
    raise RuntimeError(f"pip replaced preinstalled packages: {preinstalled} -> {after_install}")
print(f"trailcam {version('trailcam')} from commit {head}; kept {after_install}")

## 3. Check the installed code and the GPU, unpack the cache and check it

`trailcam` must not be imported yet in this kernel (Python would keep an earlier commit's
modules), and every module of the installed package must match the clone byte for byte.

Then the final scoring must not already be recorded on `main`: `origin/main` is fetched into the
clone (so a clone reused in one session is current too), and its `docs/test-ledger.md`, read with
`score_test.py`'s own `read_test_ledger`, must hold no row. A fresh clone at `COMMIT` holds that
commit's empty test ledger, so this is what stops a second scoring once the rows are pushed to
`main` on GitHub, where a commit in a local checkout is not seen (the user's decision on review
9.03). It refuses in seconds, before the cache is unpacked.

`check_gpu` refuses a session without a GPU, with other than the one pinned device visible, or
whose device torch was not built for (a P100 under Kaggle's torch), so such a session fails here
and not in the middle of the scoring.

`unpack_cache` extracts `CACHE_ARCHIVE` into `LOCAL_CACHE` (under `/tmp`, local disk, so it is not
saved as the version's output) and the scoring reads the unpacked `cct20_512/` folder; it refuses,
before writing anything, any member that is a link or a special file, has an absolute name or
`..`, or lies outside `cct20_512/`. If `CACHE_ARCHIVE` is not a file, the `.tar` files under
`/kaggle/input` are listed. First, if a folder holding a `day_night.json` is still under
`/kaggle/input` (the loose 6.03 dataset attached too), it refuses in an interactive session and
warns and continues in a committed one, as the training notebook does.

The cache must hold the annotations, the images and the day/night sidecar. The metadata is loaded
once and shown as counts only, then `check_images` refuses the cache unless the image of every
kept record of the five splits is under `images/`, with counts only, never a file name or an id
(R03). This checks that the files exist; no image is opened.

In [ ]:
if "trailcam" in sys.modules:  # imported earlier in this kernel: maybe another commit's code
    raise RuntimeError(
        "trailcam is already imported in this kernel and would not be reloaded from the clone: "
        "restart the session (Run > Factory reset) and run again"
    )
import trailcam  # noqa: E402
from trailcam.data.cache import (  # noqa: E402
    archive_files,
    check_images,
    sidecar_folders,
    unpack_cache,
)
from trailcam.data.metadata import DAY_NIGHT_SIDECAR, load_metadata  # noqa: E402
from trailcam.demo.predict import PHOTO_SUFFIXES  # noqa: E402
from trailcam.evaluate.score_test import (  # noqa: E402
    ALARM_EXIT,
    Refusal,
    check_approver,
    check_run_seeds,
    ledger_line,
    read_ledger,
    read_test_ledger,
    resolve_checkpoints,
    scoring_header,
)
from trailcam.train.device import check_gpu  # noqa: E402
from trailcam.train.train import CACHE_ANNOTATIONS, CACHE_IMAGES  # noqa: E402

installed = Path(trailcam.__file__).parent
modules = sorted(p.relative_to(CLONE / "trailcam") for p in (CLONE / "trailcam").rglob("*.py"))
differ = [
    str(m) for m in modules if (installed / m).read_bytes() != (CLONE / "trailcam" / m).read_bytes()
]
if differ:
    raise RuntimeError(f"the installed trailcam differs from the clone in {differ}")
print(f"{len(modules)} trailcam modules at {installed} match commit {head}")

In [ ]:
subprocess.run(["git", "-C", str(CLONE), "fetch", "--quiet", "origin", "main"], check=True)
MAIN_TEST_LEDGER.write_bytes(
    subprocess.run(
        ["git", "-C", str(CLONE), "show", "origin/main:docs/test-ledger.md"],
        check=True,
        capture_output=True,
    ).stdout
)
recorded = read_test_ledger(MAIN_TEST_LEDGER)
if recorded:
    raise RuntimeError(
        f"origin/main's docs/test-ledger.md already holds {len(recorded)} rows "
        f"({', '.join(row['run id'] for row in recorded)}): the final scoring has already been "
        "recorded on main, and it runs once. Nothing has been scored in this session"
    )
print("origin/main's docs/test-ledger.md holds no row: the final scoring is not recorded yet")

In [ ]:
print(check_gpu())
print(f"{os.cpu_count()} CPUs")

In [ ]:
loose = sidecar_folders(INPUT)  # the loose 6.03 dataset, attached next to the archive
run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown")  # Interactive or Batch
print(f"run type {run_type}")
if loose:
    attached = (
        f"{len(loose)} folder(s) under {INPUT} hold a {DAY_NIGHT_SIDECAR}: a loose copy of the "
        "cache (the cct20-512 dataset, or the cct20-512-tar one Kaggle extracted) is still "
        "attached, which the archive replaces. Remove it from the inputs"
    )
    if run_type == "Interactive":
        raise RuntimeError(f"{attached}, restart the session and run again")
    print(f"{run_type} run, continuing (its start is paid for): {attached}", file=sys.stderr)
if not CACHE_ARCHIVE.is_file():
    found = [str(path) for path in archive_files(INPUT)]
    raise FileNotFoundError(
        f"CACHE_ARCHIVE {CACHE_ARCHIVE} is not a file; .tar files under {INPUT}: "
        f"{found or 'none (is the output of the packing notebook attached?)'}"
    )
print(f"unpacking {CACHE_ARCHIVE} into {LOCAL_CACHE}", flush=True)
unpacked = unpack_cache(CACHE_ARCHIVE, LOCAL_CACHE)
print(unpacked)
data_root = unpacked.root  # the cache check and the scoring read the local copy

In [ ]:
needed = {
    "annotations": data_root / CACHE_ANNOTATIONS,
    "images": data_root / CACHE_IMAGES,
    "day/night sidecar": data_root / DAY_NIGHT_SIDECAR,
}
missing = [name for name, path in needed.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"the unpacked cache {data_root} lacks the {', '.join(missing)}: is CACHE_ARCHIVE the "
        "archive of the packing notebook (item 6.08)?"
    )
meta = load_metadata(needed["annotations"], needed["day/night sidecar"])
print(meta)
print(f"{sum(p.is_file() for p in needed['images'].rglob('*'))} files under {CACHE_IMAGES}/")
print(check_images(meta, needed["images"]))

## 4. Find the checkpoints and check them against the ledger

With `CHECKPOINTS_ROOT = None`, the checkpoints' folder is the one folder named
`CHECKPOINTS_DATASET` under `/kaggle/input`, at any depth, links followed (Kaggle has mounted
datasets at `/kaggle/input/datasets/<user>/<slug>`); none or several is refused, listing the
folders that hold `.pt` files. Every `.pt` file under it is a candidate.

Then `score_test.py`'s own checks, the ones its scoring runs first, called here so that a missing
or wrong checkpoint fails the rehearsal and not the final run. `resolve_checkpoints` hashes each
file and refuses, before anything is loaded: a test ledger that already holds a row (the scoring
has run); a training ledger that does not hold exactly the reported runs for `RUN_SEEDS` (a small
CNN row, and an arm A and an arm B row for each seed and no other); two rows of one checkpoint or
of one config and run seed; a file whose sha256 has no row in the ledger; a ledger row whose
checkpoint is not among the files; a config in the clone's `config.yaml` that no longer matches
its row. No weights are read here: the scoring loads the checkpoints, after the go-ahead, and
repeats every check first.

The Grad-CAM overlays are drawn from the photos directly under the clone's `assets/` (the demo's
examples; none until item 8.05), one per photo and checkpoint, titled with the predicted class.

In [ ]:
if CHECKPOINTS_ROOT is None:  # the dataset's folder, wherever Kaggle mounts it under INPUT
    named = sorted(
        {
            Path(folder).resolve()
            for folder, _, _ in os.walk(INPUT, followlinks=True)
            if Path(folder).name == CHECKPOINTS_DATASET
        }
    )
    if len(named) != 1:
        holding = sorted({str(path.parent) for path in INPUT.rglob("*.pt")})
        raise FileNotFoundError(
            f"{len(named)} folders named {CHECKPOINTS_DATASET} under {INPUT}, not one: set "
            "CHECKPOINTS_ROOT to the folder of the checkpoints; folders holding .pt files: "
            f"{holding or 'none (is the dataset attached?)'}"
        )
    checkpoint_dir = named[0]
else:
    checkpoint_dir = Path(CHECKPOINTS_ROOT)
if not checkpoint_dir.is_dir():
    raise FileNotFoundError(f"CHECKPOINTS_ROOT {checkpoint_dir} is not a folder")
checkpoint_files = sorted(checkpoint_dir.rglob("*.pt"))
print(f"{len(checkpoint_files)} .pt files under {checkpoint_dir}")

In [ ]:
checked = resolve_checkpoints(
    checkpoint_files,
    read_ledger(LEDGER),
    read_test_ledger(TEST_LEDGER),
    CONFIG_FILE,
    check_run_seeds(RUN_SEEDS),
)
for checkpoint in checked:
    print(f"{checkpoint.label}: run {checkpoint.run_id}, sha256 {checkpoint.sha256}")
print(
    f"{len(checked)} checkpoints, one per ledger row, each file's sha256 that of its row; the "
    "test ledger holds no row; no checkpoint loaded"
)
overlay_photos = sorted(path for path in ASSETS.glob("*") if path.suffix.lower() in PHOTO_SUFFIXES)
print(f"{len(overlay_photos)} assets/ photos for the Grad-CAM overlays")

## 5. The go-ahead

The final scoring needs the user's go-ahead, recorded by name in every test ledger row. While
`APPROVED_BY` is empty this cell stops the run, with that message, before anything is loaded:
every check above has passed by then, so a Run All with `APPROVED_BY` empty is the rehearsal. It
uses `score_test.py`'s own `check_approver`, which also refuses a name holding a `|`, a line break
or another non-printable character. `score_test.py` checks `--approved-by` again, so the scoring
cell run on its own with `APPROVED_BY` empty is refused too.

Never Save & Run All with `APPROVED_BY` filled in until the user means it.

In [ ]:
try:
    approver = check_approver(APPROVED_BY)
except Refusal as refusal:
    raise RuntimeError(
        f"APPROVED_BY {APPROVED_BY!r} is refused: {refusal}. The final scoring needs the user's "
        "go-ahead. Every check above has passed and nothing has been scored. Fill in APPROVED_BY "
        "in the parameter cell only when the user means the one final scoring"
    ) from None
print(f"approved by {approver}: section 6 runs the one final scoring")

## 6. The scoring

One call of `python -m trailcam.evaluate.score_test`, in its own Python process, given every
checkpoint checked in section 4, the unpacked cache, the clone's config, ledger and test ledger,
`OUT`, and one overlay per `assets/` photo. It checks everything again before anything is loaded,
draws the overlays, scores each checkpoint once on cis_test and once on trans_test, prints that
checkpoint's test ledger row as soon as it is scored, writes the tables and figures under `OUT`,
and appends every row to the clone's test ledger last. Its output is shown here line by line.
The command is printed with the `assets/` photos counted, never named: a photo's name may be an
image id (R03).

Exit 0: done. Exit 3 (`ALARM_EXIT`): done, and a leakage alarm fired on the test scores (R16); the
rows are appended, and the notebook goes on to print them and zip the outputs, since the scoring
cannot be run again; then stop and report the alarm before anything else. Any other exit fails
this cell: the reason is printed above it, and "If it stops part way" says what to do.

In [ ]:
import shlex

argv = [f"--approved-by={APPROVED_BY}", "--run-seeds", *map(str, RUN_SEEDS)]
argv += [arg for checkpoint in checked for arg in ("--checkpoint", str(checkpoint.path))]
argv += ["--data-root", str(data_root), "--config", str(CONFIG_FILE), "--ledger", str(LEDGER)]
argv += ["--test-ledger", str(TEST_LEDGER), "--out", str(OUT), "--assets", str(ASSETS)]
command = [sys.executable, "-m", "trailcam.evaluate.score_test", *argv]
overlays = [arg for photo in overlay_photos for arg in ("--overlay", str(photo))]
shown = f" and {len(overlay_photos)} --overlay photos of assets/" if overlay_photos else ""
print(shlex.join(command) + shown, flush=True)  # R03: the photos counted, never named
with subprocess.Popen(
    [*command, *overlays],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
) as scoring:
    for line in scoring.stdout:
        print(line, end="", flush=True)
exit_code = scoring.returncode
if exit_code == ALARM_EXIT:
    print("LEAKAGE ALARM (R16): the alarms are printed above; report them", file=sys.stderr)
elif exit_code != 0:
    raise RuntimeError(
        f"the scoring exited {exit_code}; the reason is printed above. If it did not print "
        "'appended ... rows', nothing was appended: run again from the start with a new OUT "
        "(How to run, If it stops part way)"
    )

## 7. The test ledger rows, and the outputs zipped

The rows the scoring appended to the clone's test ledger, read back with `score_test.py`'s reader
and checked against section 4: one row per checkpoint, by sha256. Paste them once into
`docs/test-ledger.md`, below its header (the header printed here is the one already there).

Then `OUT` is zipped into `score.zip` next to it, kept as the version's output: `headline.md`,
`slices.md` and the feature maps, and per run id the metrics tables and figures, the attention
tables and histogram, and the overlays.

In [ ]:
rows = read_test_ledger(TEST_LEDGER)
if sorted(row["checkpoint sha256"] for row in rows) != sorted(c.sha256 for c in checked):
    raise RuntimeError(
        f"{TEST_LEDGER} holds {len(rows)} rows, not one per checkpoint of section 4 "
        f"({len(checked)}): the scoring did not finish (How to run, If it stops part way)"
    )
print(f"commit {head}: {len(rows)} rows, one per checkpoint")
print("test ledger rows, for docs/test-ledger.md (paste once, below its header):")
print(scoring_header())
for row in rows:
    print(ledger_line(row))

In [ ]:
import shutil

archive = Path(shutil.make_archive(str(OUT), "zip", root_dir=OUT.parent, base_dir=OUT.name))
files = sum(path.is_file() for path in OUT.rglob("*"))
print(f"{archive}: the {files} files under {OUT}, {archive.stat().st_size / 2**20:.1f} MiB")
print("download it from the version's Output, or from the session's Output panel")
if exit_code == ALARM_EXIT:
    print("LEAKAGE ALARM (R16): stop and report the alarms before anything else", file=sys.stderr)